In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import json, pickle, warnings

In [2]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import (RandomForestRegressor,
                              GradientBoostingRegressor,
                              ExtraTreesRegressor)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error,
                             mean_squared_error, 
                             r2_score)

In [3]:
plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d27'
plt.rcParams['text.color']       = '#e0e0e0'
plt.rcParams['axes.labelcolor']  = '#e0e0e0'
plt.rcParams['xtick.color']      = '#9e9e9e'
plt.rcParams['ytick.color']      = '#9e9e9e'
C_BLUE  = '#6366f1'; C_GREEN = '#10b981'
C_AMBER = '#f59e0b'; C_RED   = '#ef4444'



In [4]:
# Load Data & Configure
import json
df = pd.read_csv('C:/Users/HP/Downloads/edunet-IBM AIML project/retail_features.csv', parse_dates=['date'],low_memory=False)
with open('C:/Users/HP/Downloads/edunet-IBM AIML project/feature_config.json') as f:
    config = json.load(f)

FEATURES = config['features']
TARGET   = config['target']

print(f'Dataset  : {df.shape}')
print(f'Features : {len(FEATURES)}')
print(f'Dates    : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Demand   : min={df[TARGET].min()}  max={df[TARGET].max()}  mean={df[TARGET].mean():.1f}')

Dataset  : (22420, 55)
Features : 41
Dates    : 2009-12-01 → 2011-12-09
Demand   : min=0  max=80995  mean=79.7


In [5]:
print('\nRemoving burn-in rows (first 28 per product)...')
df['row_num'] = df.groupby('stockcode').cumcount()
df_model = df[df['row_num'] >= 28].copy()
print(f'Rows before: {len(df):,}')
print(f'Rows after : {len(df_model):,}  (removed first 28 per product)')



Removing burn-in rows (first 28 per product)...
Rows before: 22,420
Rows after : 21,860  (removed first 28 per product)


## Time-Based Train - Validation - Test Split

In [6]:
print('\nCreating time-based split...')

total_days = (df_model['date'].max() - df_model['date'].min()).days
train_end  = df_model['date'].min() + pd.Timedelta(days=int(total_days * 0.70))
val_end    = df_model['date'].min() + pd.Timedelta(days=int(total_days * 0.85))

train_df = df_model[df_model['date'] <= train_end]
val_df   = df_model[(df_model['date'] > train_end) & (df_model['date'] <= val_end)]
test_df  = df_model[df_model['date'] > val_end]

X_train = train_df[FEATURES].fillna(0);  y_train = train_df[TARGET]
X_val   = val_df[FEATURES].fillna(0);    y_val   = val_df[TARGET]
X_test  = test_df[FEATURES].fillna(0);   y_test  = test_df[TARGET]

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f'Train : {len(train_df):,} rows')
print(f'Val   : {len(val_df):,} rows')
print(f'Test  : {len(test_df):,} rows')




Creating time-based split...
Train : 15,245 rows
Val   : 3,294 rows
Test  : 3,321 rows


## Evaluation Function

In [7]:
def evaluate(y_true, y_pred, name):
    """
    Computes metrics and returns a DICTIONARY.
    Always call as:  results.append( evaluate(y_val, preds, name) )
    """
    y_true = np.array(y_true)
    y_pred = np.maximum(0, np.array(y_pred))

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs(y_true - y_pred) / (np.abs(y_true) + 1)) * 100
    acc  = max(0, 100 - mape)

    print(f'\n  {name}')
    print(f'    MAE      : {mae:.2f} units')
    print(f'    RMSE     : {rmse:.2f} units')
    print(f'    R²       : {r2:.4f}')
    print(f'    MAPE     : {mape:.2f}%')
    print(f'    Accuracy : {acc:.2f}%')

    # MUST return a dict — results.append() needs this
    return {
        'model'   : name,
        'MAE'     : round(float(mae),  4),
        'RMSE'    : round(float(rmse), 4),
        'R2'      : round(float(r2),   4),
        'MAPE'    : round(float(mape), 4),
        'Accuracy': round(float(acc),  4),
    }



In [8]:
results = []   # ← start with empty list
trained = {}   # ← stores model objects

## Train All Models

In [9]:
# 1. Ridge Regression
print('\n[1/4] Ridge Regression...')
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_s, y_train)
ridge_preds = np.maximum(0, ridge.predict(X_val_s))
results.append( evaluate(y_val, ridge_preds, 'Ridge') )   # ← correct
trained['Ridge'] = ridge


[1/4] Ridge Regression...

  Ridge
    MAE      : 61.10 units
    RMSE     : 177.53 units
    R²       : 0.1834
    MAPE     : 692.33%
    Accuracy : 0.00%


In [10]:
# 2. Random Forest
print('\n[2/4] Random Forest...')
rf = RandomForestRegressor(n_estimators=100, max_depth=12,
                            min_samples_split=5, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
rf_preds = np.maximum(0, rf.predict(X_val))
results.append( evaluate(y_val, rf_preds, 'Random Forest') )   # ← correct
trained['Random Forest'] = rf


[2/4] Random Forest...

  Random Forest
    MAE      : 36.47 units
    RMSE     : 244.84 units
    R²       : -0.5533
    MAPE     : 103.96%
    Accuracy : 0.00%


In [11]:
# 3. Gradient Boosting
print('\n[3/4] Gradient Boosting...')
gb = GradientBoostingRegressor(n_estimators=150, max_depth=5,
                                learning_rate=0.1, subsample=0.8, random_state=42)
gb.fit(X_train, y_train)
gb_preds = np.maximum(0, gb.predict(X_val))
results.append( evaluate(y_val, gb_preds, 'Gradient Boosting') )   # ← correct
trained['Gradient Boosting'] = gb



[3/4] Gradient Boosting...

  Gradient Boosting
    MAE      : 34.74 units
    RMSE     : 207.71 units
    R²       : -0.1178
    MAPE     : 224.33%
    Accuracy : 0.00%


In [12]:
# 4. Extra Trees
print('\n[4/4] Extra Trees...')
et = ExtraTreesRegressor(n_estimators=100, max_depth=12,
                          n_jobs=-1, random_state=42)
et.fit(X_train, y_train)
et_preds = np.maximum(0, et.predict(X_val))
results.append( evaluate(y_val, et_preds, 'Extra Trees') )   # ← correct
trained['Extra Trees'] = et


[4/4] Extra Trees...

  Extra Trees
    MAE      : 40.15 units
    RMSE     : 194.53 units
    R²       : 0.0195
    MAPE     : 111.01%
    Accuracy : 0.00%


In [13]:
print(f'\n\nresults list has {len(results)} entries (should be 4)')
print(f'First entry: {results[0]}')



results list has 4 entries (should be 4)
First entry: {'model': 'Ridge', 'MAE': 61.1025, 'RMSE': 177.5315, 'R2': 0.1834, 'MAPE': 692.3283, 'Accuracy': 0.0}


## Compare Models & Pick Best

In [14]:
results_df = (pd.DataFrame(results)
                .sort_values('RMSE')
                .reset_index(drop=True))

print('\n' + '='*60)
print('MODEL COMPARISON (sorted by RMSE, lower = better)')
print('='*60)
print(results_df[['model','MAE','RMSE','R2','MAPE','Accuracy']]
      .round(2)
      .to_string(index=False))

best_name  = results_df.iloc[0]['model']
best_model = trained[best_name]

print(f'\nBEST MODEL : {best_name}')
print(f'R²         : {results_df.iloc[0]["R2"]:.4f}')
print(f'Accuracy   : {results_df.iloc[0]["Accuracy"]:.1f}%')
print(f'MAPE       : {results_df.iloc[0]["MAPE"]:.2f}%')


MODEL COMPARISON (sorted by RMSE, lower = better)
            model   MAE   RMSE    R2   MAPE  Accuracy
            Ridge 61.10 177.53  0.18 692.33       0.0
      Extra Trees 40.15 194.53  0.02 111.01       0.0
Gradient Boosting 34.74 207.71 -0.12 224.33       0.0
    Random Forest 36.47 244.84 -0.55 103.96       0.0

BEST MODEL : Ridge
R²         : 0.1834
Accuracy   : 0.0%
MAPE       : 692.33%


In [15]:
pkg = {
    'model'      : best_model,
    'model_name' : best_name,
    'features'   : FEATURES,
    'target'     : TARGET,
    'scaler'     : scaler,
    'val_results': results,
    'config'     : config,
    'train_end'  : str(train_end.date()),
    'val_end'    : str(val_end.date()),
}
with open('C:/Users/HP/Downloads/edunet-IBM AIML project/best_model.pkl', 'wb') as f:
    pickle.dump(pkg, f)

print(f'\nSaved: best_model.pkl')




Saved: best_model.pkl


In [16]:
## Feature Importance
if hasattr(best_model, 'feature_importances_'):
    imp = pd.DataFrame({
        'feature'   : FEATURES,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=True).tail(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = [C_AMBER if v > imp['importance'].quantile(0.8) else C_BLUE
              for v in imp['importance']]
    ax.barh(imp['feature'], imp['importance'], color=colors, alpha=0.85)
    ax.set_title(f'Top 20 Feature Importances — {best_name}',
                 fontsize=12, fontweight='bold', color='white')
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig('C:/Users/HP/Downloads/edunet-IBM AIML project/chart_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nTop 5 Most Important Features:')
    top5 = pd.DataFrame({'feature':FEATURES,
                          'importance':best_model.feature_importances_})\
             .nlargest(5,'importance')
    for _, row in top5.iterrows():
        bar = '█' * int(row['importance']*300)
        print(f'   {row["feature"]:<30} {bar}  {row["importance"]:.4f}')

## Actual vs Predicted Chart

In [17]:
best_vp = val_pred[best_name]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'{best_name} — Prediction Quality',
             fontsize=12, fontweight='bold', color='white')

idx = np.random.choice(len(y_val), min(500, len(y_val)), replace=False)
axes[0].scatter(y_val.values[idx], best_vp[idx],
                alpha=0.4, s=15, color=C_BLUE)
mv = max(y_val.max(), best_vp.max())
axes[0].plot([0,mv],[0,mv],'r--', linewidth=2, label='Perfect fit')
axes[0].set_xlabel('Actual Demand')
axes[0].set_ylabel('Predicted Demand')
axes[0].set_title('Actual vs Predicted', fontweight='bold', color='white')
axes[0].legend()

residuals = y_val.values - best_vp
axes[1].hist(residuals, bins=50, color=C_GREEN, alpha=0.8, edgecolor='none')
axes[1].axvline(0, color='white', linestyle='--', linewidth=2,
                label='Zero error')
axes[1].axvline(residuals.mean(), color=C_AMBER, linewidth=2,
                label=f'Mean: {residuals.mean():.2f}')
axes[1].set_xlabel('Residual (Actual − Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals — should centre at 0',
                  fontweight='bold', color='white')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/chart_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data/chart_actual_vs_predicted.png')

NameError: name 'val_pred' is not defined

In [ ]:
pkg = {
    'model'      : best_model,
    'model_name' : best_name,
    'features'   : FEATURES,
    'target'     : TARGET,
    'scaler'     : scaler,
    'val_results': results,
    'config'     : config,
    'train_end'  : str(train_end.date()),
    'val_end'    : str(val_end.date()),
}
with open('models/best_model.pkl','wb') as f:
    pickle.dump(pkg, f)

print(f'\nmodels/best_model.pkl saved')
print(f'Best model : {best_name}')
print(f'R²         : {results_df.iloc[0]["R2"]:.4f}')
print(f'Accuracy   : {results_df.iloc[0]["Accuracy"]:.1f}%')
